# 02. Market Construction — Building Exotic Betting Markets

Construct derived betting markets from inning-by-inning data:
- **YRFI** (Yes Run First Inning)
- **F5** (First 5 Innings) results and totals
- **Run Line** cover rates
- **Race to X** runs
- **Over/Under** analysis

Validate market construction logic and establish baseline rates.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

from src.data_loader import load_all_seasons, apply_data_filters, apply_betting_filters, add_derived_odds
from src.market_builder import add_all_markets, market_summary

# Load and prepare data
games = load_all_seasons()
games = apply_data_filters(games)
games = add_derived_odds(games)
bettable = apply_betting_filters(games)
bettable = add_all_markets(bettable)

print(f'Bettable games with all markets: {len(bettable)}')
print(f'Columns: {len(bettable.columns)}')
print()
print('Market summary:')
summary = market_summary(bettable)
for k, v in summary.items():
    print(f'  {k}: {v:.4f}')

## 1. YRFI (Yes Run First Inning) Market

In [ ]:
# YRFI deep dive
print(f'YRFI Rate: {bettable["yrfi"].mean():.1%}')
print(f'NRFI Rate: {1 - bettable["yrfi"].mean():.1%}')
print(f'Avg 1st inning runs: {bettable["first_inning_runs"].mean():.3f}')
print()

# YRFI breakdown: who scores
both_score = ((bettable['away_inn_1'] > 0) & (bettable['home_inn_1'] > 0)).mean()
only_away = ((bettable['away_inn_1'] > 0) & (bettable['home_inn_1'] == 0)).mean()
only_home = ((bettable['away_inn_1'] == 0) & (bettable['home_inn_1'] > 0)).mean()
neither = ((bettable['away_inn_1'] == 0) & (bettable['home_inn_1'] == 0)).mean()

print('1st Inning Scoring Breakdown:')
print(f'  Both teams score: {both_score:.1%}')
print(f'  Only away scores: {only_away:.1%}')
print(f'  Only home scores: {only_home:.1%}')
print(f'  Neither scores:   {neither:.1%} (= NRFI rate)')

# YRFI by season
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
yrfi_season = bettable.groupby('season')['yrfi'].mean()
ax.bar(yrfi_season.index.astype(int), yrfi_season.values * 100, color='steelblue', alpha=0.8)
ax.axhline(bettable['yrfi'].mean() * 100, color='red', linestyle='--', linewidth=2,
           label=f'Overall: {bettable["yrfi"].mean():.1%}')
ax.set_xlabel('Season')
ax.set_ylabel('YRFI Rate (%)')
ax.set_title('YRFI Rate by Season')
ax.legend()

# First inning run distribution
ax = axes[1]
ax.hist(bettable['first_inning_runs'], bins=range(0, 10), color='#ff7043', alpha=0.8, edgecolor='white')
ax.set_xlabel('Total 1st Inning Runs')
ax.set_ylabel('Count')
ax.set_title('1st Inning Runs Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# YRFI ROI at fixed 1.80 odds
yrfi_rate = bettable['yrfi'].mean()
yrfi_odds = 1.80

# EV per $1 bet on YRFI
ev_yrfi = yrfi_rate * (yrfi_odds - 1) - (1 - yrfi_rate)
# EV per $1 bet on NRFI
nrfi_odds = 1 / (1 - 1/yrfi_odds)  # Approximate complement
ev_nrfi = (1 - yrfi_rate) * (nrfi_odds - 1) - yrfi_rate

print(f'YRFI at 1.80 odds:')
print(f'  Win rate: {yrfi_rate:.1%}')
print(f'  EV per $1: ${ev_yrfi:.3f}')
print(f'  ROI: {ev_yrfi*100:.1f}%')

# Simulate flat betting
n_bets = len(bettable)
wins = bettable['yrfi'].sum()
total_pnl = wins * (yrfi_odds - 1) * 100 - (n_bets - wins) * 100
roi = total_pnl / (n_bets * 100)
print(f'\nFlat $100 YRFI betting:')
print(f'  Bets: {n_bets}, Wins: {int(wins)}')
print(f'  P&L: ${total_pnl:,.0f}')
print(f'  ROI: {roi:.1%}')
print(f'\nNote: Break-even YRFI rate at 1.80 odds = {1/1.80:.1%}')
print(f'      Actual YRFI rate = {yrfi_rate:.1%} -> {"PROFITABLE" if yrfi_rate > 1/1.80 else "UNPROFITABLE"} at 1.80')

## 2. F5 (First 5 Innings) Market

In [ ]:
print('F5 Market Analysis:')
print(f'  Avg F5 total: {bettable["f5_total"].mean():.2f}')
print(f'  F5 Home Win: {bettable["f5_home_win"].mean():.1%}')
print(f'  F5 Push (tie): {(bettable["f5_winner"] == "push").mean():.1%}')
print(f'  F5 Away Win: {(bettable["f5_winner"] == "away").mean():.1%}')
print(f'  F5 Over rate (half O/U line): {bettable["f5_over"].mean():.1%}')
print(f'  Avg F5 O/U proxy line: {bettable["f5_ou_line"].mean():.2f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# F5 total distribution
ax = axes[0]
ax.hist(bettable['f5_total'], bins=range(0, 18), color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(bettable['f5_total'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Mean: {bettable["f5_total"].mean():.1f}')
ax.set_xlabel('F5 Total Runs')
ax.set_ylabel('Count')
ax.set_title('F5 Total Runs Distribution')
ax.legend()

# F5 winner pie chart
ax = axes[1]
f5_results = bettable['f5_winner'].value_counts()
ax.pie(f5_results.values, labels=f5_results.index, autopct='%1.1f%%',
       colors=['#388e3c', '#d32f2f', '#757575'])
ax.set_title('F5 Winner Distribution')

# F5 home lead distribution
ax = axes[2]
f5_lead = bettable['home_f5_runs'] - bettable['away_f5_runs']
ax.hist(f5_lead, bins=range(-10, 11), color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(0, color='red', linestyle='-', linewidth=2)
ax.set_xlabel('F5 Home Lead (positive = home leading)')
ax.set_ylabel('Count')
ax.set_title('F5 Home Lead Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# F5 result vs full game result correlation
# How often does the F5 leader win the game?
bt = bettable.copy()
bt['full_winner'] = np.where(bt['home_win'] == 1, 'home', 'away')
bt['f5_correct'] = bt['f5_winner'] == bt['full_winner']

# Only non-push F5 results
non_push = bt[bt['f5_winner'] != 'push']
f5_predictive = non_push['f5_correct'].mean()

print(f'F5 Leader wins the game: {f5_predictive:.1%} (of non-push F5 results)')
print(f'F5 Push rate: {(bt["f5_winner"] == "push").mean():.1%}')
print()
print('Implication: F5 result is a strong predictor of full-game outcome.')
print('F5 bets are valuable because they remove late-game variance (bullpen, etc).')

## 3. Run Line Market

In [ ]:
print('Run Line Analysis:')
print(f'  Home covers -1.5: {bettable["home_rl_cover"].mean():.1%}')
print(f'  Away covers +1.5: {bettable["away_rl_cover"].mean():.1%}')
print(f'  1-run games (neither covers): {1 - bettable["home_rl_cover"].mean() - bettable["away_rl_cover"].mean():.1%}')

# Run line by favorite/underdog
fav = bettable[bettable['home_is_favorite']]
dog = bettable[~bettable['home_is_favorite']]
print(f'\n  Home favorite covers -1.5: {fav["home_rl_cover"].mean():.1%} (n={len(fav)})')
print(f'  Home underdog covers -1.5: {dog["home_rl_cover"].mean():.1%} (n={len(dog)})')

# Margin distribution
fig, ax = plt.subplots(figsize=(14, 5))
margin = bettable['home_margin']
ax.hist(margin, bins=range(-15, 16), color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(1.5, color='red', linestyle='--', linewidth=2, label='Run Line +1.5/-1.5')
ax.axvline(-1.5, color='red', linestyle='--', linewidth=2)
ax.axvline(0, color='gray', linestyle='-', linewidth=1)
ax.set_xlabel('Home Margin (positive = home win)')
ax.set_ylabel('Count')
ax.set_title(f'Run Margin Distribution (Home RL cover: {bettable["home_rl_cover"].mean():.1%})')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Race to X Runs Market

In [ ]:
# Race to 3 analysis
print('Race to 3 Runs:')
r2_3 = bettable['race_to_3'].value_counts(normalize=True)
for result, pct in r2_3.items():
    print(f'  {result}: {pct:.1%}')

# Race to different X values
print('\nRace to X comparison (% of games):')
from src.market_builder import add_race_to_x_market
for x in [1, 2, 3, 4, 5]:
    bt = add_race_to_x_market(bettable, x=x)
    col = f'race_to_{x}'
    home_pct = (bt[col] == 'home').mean()
    away_pct = (bt[col] == 'away').mean()
    none_pct = (bt[col] == 'none').mean()
    print(f'  Race to {x}: Home={home_pct:.1%}, Away={away_pct:.1%}, None={none_pct:.1%}')

## 5. Over/Under Market

In [ ]:
bt = bettable.copy()
bt['over'] = (bt['total_runs'] > bt['close_ou']).astype(int)
bt['under'] = (bt['total_runs'] < bt['close_ou']).astype(int)

print(f'Over/Under Analysis:')
print(f'  Over rate: {bt["over"].mean():.1%}')
print(f'  Under rate: {bt["under"].mean():.1%}')
print(f'  Push rate: {1 - bt["over"].mean() - bt["under"].mean():.1%}')
print(f'  Avg O/U line: {bt["close_ou"].mean():.2f}')
print(f'  Avg actual total: {bt["total_runs"].mean():.2f}')

# Trend over seasons
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
by_season = bt.groupby('season').agg(
    avg_line=('close_ou', 'mean'),
    avg_actual=('total_runs', 'mean'),
    over_rate=('over', 'mean'),
)
ax.plot(by_season.index, by_season['avg_line'], 'o-', color='steelblue', linewidth=2, label='Avg O/U Line')
ax.plot(by_season.index, by_season['avg_actual'], 's-', color='#d32f2f', linewidth=2, label='Avg Actual Total')
ax.set_xlabel('Season')
ax.set_ylabel('Runs')
ax.set_title('O/U Line vs Actual Total by Season')
ax.legend()

ax = axes[1]
ax.bar(by_season.index.astype(int), by_season['over_rate'] * 100, color='#ff7043', alpha=0.8)
ax.axhline(50, color='gray', linestyle='--', linewidth=1.5)
ax.set_xlabel('Season')
ax.set_ylabel('Over Rate (%)')
ax.set_title('Over Hit Rate by Season')

plt.tight_layout()
plt.show()

## 6. Cross-Market Correlations

In [ ]:
# Do YRFI games tend to be higher scoring overall?
yrfi_games = bettable[bettable['yrfi'] == 1]
nrfi_games = bettable[bettable['yrfi'] == 0]

print('Cross-Market Correlations:')
print(f'\n1. YRFI games vs NRFI games:')
print(f'  YRFI avg total runs: {yrfi_games["total_runs"].mean():.2f}')
print(f'  NRFI avg total runs: {nrfi_games["total_runs"].mean():.2f}')
print(f'  YRFI F5 avg total: {yrfi_games["f5_total"].mean():.2f}')
print(f'  NRFI F5 avg total: {nrfi_games["f5_total"].mean():.2f}')
print(f'  YRFI home RL cover: {yrfi_games["home_rl_cover"].mean():.1%}')
print(f'  NRFI home RL cover: {nrfi_games["home_rl_cover"].mean():.1%}')

# Correlation matrix
corr_cols = ['yrfi', 'f5_total', 'f5_home_win', 'home_rl_cover', 'home_win',
             'total_runs', 'home_race_to_3']
corr_matrix = bettable[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax, vmin=-0.5, vmax=0.5)
ax.set_title('Market Cross-Correlations')
plt.tight_layout()
plt.show()

## 7. Favorite Performance Across Markets

In [ ]:
# How do favorites perform across different markets?
bt = bettable.copy()

# Bin by implied probability
bt['fav_strength'] = pd.cut(bt['home_implied_prob'], 
                            bins=[0.3, 0.45, 0.50, 0.55, 0.60, 0.65, 0.75],
                            labels=['Heavy dog', 'Slight dog', 'Toss-up', 'Slight fav', 'Mod fav', 'Heavy fav'])

fav_perf = bt.groupby('fav_strength', observed=True).agg(
    n=('home_win', 'count'),
    ml_wr=('home_win', 'mean'),
    rl_cover=('home_rl_cover', 'mean'),
    f5_wr=('f5_home_win', 'mean'),
    yrfi=('yrfi', 'mean'),
    race3_home=('home_race_to_3', 'mean'),
).reset_index()

print('Home team performance by favorite strength:')
print(fav_perf.to_string(index=False))

## 8. Summary & Baseline Rates

In [ ]:
print('=' * 70)
print('MARKET CONSTRUCTION SUMMARY — Baseline Rates')
print('=' * 70)
bt = bettable.copy()
bt['over'] = (bt['total_runs'] > bt['close_ou']).astype(int)
print(f'''
Bettable games: {len(bettable)}

YRFI (Yes Run First Inning):
  Rate: {bettable['yrfi'].mean():.1%}
  Break-even at 1.80: {1/1.80:.1%}
  Edge: {(bettable['yrfi'].mean() - 1/1.80)*100:+.1f}%

F5 (First 5 Innings):
  Avg total: {bettable['f5_total'].mean():.2f}
  Home win: {bettable['f5_home_win'].mean():.1%}
  Push: {(bettable['f5_winner'] == 'push').mean():.1%}

Run Line (-1.5):
  Home cover: {bettable['home_rl_cover'].mean():.1%}
  Away cover: {bettable['away_rl_cover'].mean():.1%}

Race to 3:
  Home first: {bettable['home_race_to_3'].mean():.1%}
  Away first: {(bettable['race_to_3'] == 'away').mean():.1%}
  Neither: {(bettable['race_to_3'] == 'none').mean():.1%}

Over/Under:
  Over rate: {bt['over'].mean():.1%}
  Avg line: {bt['close_ou'].mean():.1f}
  Avg actual: {bt['total_runs'].mean():.1f}

Moneyline:
  Home win: {bettable['home_win'].mean():.1%}
  Avg home implied prob: {bettable['home_implied_prob'].mean():.3f}
''')
print('All markets constructed successfully. Ready for feature engineering and backtesting.')